# Interpolation of JCOPE outputs from sigma to z coordinates

In [1]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import os
from numba import njit, prange
import pandas as pd

In [2]:
# Define the experiment directory
dir_exp = os.path.expanduser('~/data/py-off-bgc/input/JCOPE_FGO/nc_sigma/')

# Suffix for inputs
suffix_in = "*_NWP.nc"


# Define Target Z-Levels (meters)
z_targets = np.array([
    0, 5, 10, 15, 20, 25, 30, 35, 40, 45, 50, 
    60, 70, 80, 90, 100, 125, 150, 175, 200, 
    250, 300, 350, 400, 450, 500, 600, 700, 800, 900, 1000
], dtype=np.float32)

# Define your list of variables
list_var = [
#    {
#        "filename": "T",
#        "varname": "tm",
#        "units": "degC",
#        "cfname": "sea_water_temperature"
#    },
#    {
#        "filename": "S",
#        "varname": "sm",
#        "units": "psu",
#        "cfname": "sea_water_salinity"
#    },
#    {
#        "filename": "U",
#        "varname": "um",
#        "units": "m/s",
#        "cfname": "eastward_sea_water_velocity"
#    },
    {
        "filename": "V",
        "varname": "vm",
        "units": "m/s",
        "cfname": "northward_sea_water_velocity"
    },
]

### Interpolation from sigma-coordinate to z-coordinate

In [3]:
def fix_jcope_dims(ds_in):
    # Fix JCOPE dimension to be in the order (t,z,y,x).
    # Get rid of dimension(s) of size 1, which is unnecessary.
    return ds_in.transpose("time","lev","lat","lon").squeeze()

# 1. Load grid data (basic)
ds_grid = xr.open_dataset(dir_exp+'basic_NWP.nc')
ds_grid = fix_jcope_dims(ds_grid)
z_static = abs(ds_grid['zz'])  # zz values are negatives, so use abs() to convert to positives
# Bathymetry (H)
H = ds_grid['dz'].sum("lev")

# Load sea level data (EL)
ds_el = xr.open_mfdataset(dir_exp+"EL/"+suffix_in)
ds_el = fix_jcope_dims(ds_el)
el = ds_el['elm']

# Calculate ACTUAL 3D Depth for each day
#    Formula: Z_static * (1 + EL / H)
z_daily = z_static * (1.0 + el / H)
z_daily = z_daily.transpose("time","lev","lat","lon")

@njit(parallel=True)
def interpolate_fast(z3d_in, in_da, z_targets, out_da):
    nz, nj, ni = in_da.shape
    
    # Numba likes explicit loops! 
    # Use 'prange' on the outer loop to use all CPU cores automatically
    for j in prange(nj):
        for i in range(ni):
            # Extract columns for depth and data (Standard numpy slicing works in Numba)
            col_z = z3d_in[:, j, i]
            col_da = in_da[:, j, i]
            
            # Numba supports np.interp
            out_da[:, j, i] = np.interp(z_targets, col_z, col_da)

for v in list_var:
    print(f"Interpolating {v['filename']}...")

    # Read the input data
    ds_in = xr.open_mfdataset(dir_exp+v["filename"]+'/'+suffix_in)
    ds_in = fix_jcope_dims(ds_in)
    da_in = ds_in[v["varname"]]
    nt, ny, nx = da_in["time"].size, da_in["lat"].size, da_in["lon"].size
    # Initialize the output array
    da_out = np.full((nt, len(z_targets), ny, nx), np.nan, dtype=np.float32)

    for t in range(nt):
        # Call Numba
        interpolate_fast(z_daily[t,:,:,:].values, da_in[t,:,:,:].values, z_targets, da_out[t,:,:,:])
        
    # Convert back to Xarray
    out_array = xr.DataArray(
        da_out,
        coords={
            "time": ds_in['time'],       # Copy time from the input file
            "depth": z_targets,       # New Z coordinates
            "lat": ds_grid["lat"],         # Copy lat from the grid file
            "lon": ds_grid["lon"]          # Copy lon from the grid file
        },
        dims=("time", "depth", "lat", "lon"),
        name=v["varname"],
        attrs={
            "units": v["units"],
            "standard_name": v["cfname"]
        }
    )

    # Save to Disk
    out_filename = dir_exp+"interp_JCOPE-FGO_NWP_"+v['filename']+'.nc'
    
    # converting to dataset allows preserving global attributes easily
    ds_out = out_array.to_dataset()
    ds_out.to_netcdf(out_filename)
    
print ("Interpolation finished!")

Interpolating V...
Interpolation finished!


### check if the landsea mask locations are the same among the T, S, U, V data.
* it looks like the landsea mask is consistent among the dataset.

In [16]:
xr.open_mfdataset('../input/JCOPE-FGO/nc_sigma/NSWR/NSWR_*_NWP.nc')

<xarray.Dataset> Size: 160MB
Dimensions:  (time: 366, lev: 1, lat: 330, lon: 331)
Coordinates:
  * time     (time) datetime64[ns] 3kB 2024-01-01T12:00:00 ... 2024-12-31T12:...
  * lev      (lev) float64 8B 1.0
  * lat      (lat) float64 3kB 17.05 17.15 17.25 17.35 ... 49.75 49.85 49.95
  * lon      (lon) float64 3kB 117.0 117.1 117.2 117.3 ... 149.8 149.9 150.0
Data variables:
    nswrm    (time, lev, lat, lon) float32 160MB dask.array<chunksize=(1, 1, 330, 331), meta=np.ndarray>
Attributes:
    CDI:          Climate Data Interface version 2.0.4 (https://mpimet.mpg.de...
    Conventions:  CF-1.6
    history:      Mon May 25 11:57:40 2026: cdo -s -z zip_1 sellonlatbox,117,...
    CDO:          Climate Data Operators version 2.0.4 (https://mpimet.mpg.de...

In [15]:
ds_t = xr.open_dataset('/home/hakaseh/data/py-off-bgc/input/JCOPE_FGO/nc_sigma/interp_JCOPE-FGO_NWP_T.nc')
ds_u = xr.open_dataset('/home/hakaseh/data/py-off-bgc/input/JCOPE_FGO/nc_sigma/interp_JCOPE-FGO_NWP_U.nc')
ds_v = xr.open_dataset('/home/hakaseh/data/py-off-bgc/input/JCOPE_FGO/nc_sigma/interp_JCOPE-FGO_NWP_V.nc')
t = ds_t['tm'][0,0,:,:]
u = ds_u['um'][0,0,:,:]
v = ds_v['vm'][0,0,:,:]
tuv = t/t+u/u+v/v
tuv.min(),tuv.max()

(<xarray.DataArray ()> Size: 4B
 array(3., dtype=float32)
 Coordinates:
     time     datetime64[ns] 8B 2024-01-01T12:00:00
     depth    float32 4B 0.0
 Attributes:
     units:          m/s
     standard_name:  northward_sea_water_velocity,
 <xarray.DataArray ()> Size: 4B
 array(3., dtype=float32)
 Coordinates:
     time     datetime64[ns] 8B 2024-01-01T12:00:00
     depth    float32 4B 0.0
 Attributes:
     units:          m/s
     standard_name:  northward_sea_water_velocity)

### basic.nc explained
The `basic` file in the JCOPE output contains grid data.

Dimensions:
- lon: longitude (range: 0 to 360 deg)
- lat: latitude (range: -90 to 90 deg)
- lev: vertical levels in integers starting from 1
- time: time only has one value which is static.

Variables (time, lev, lat, lon):
- z: the upper depth point (negative values)
- zz: the center depth point (negative values)
- dz: the vertical resolution (positive values; approximately equal to (z - zz)*2)